# RQ2 — RNA repeat length

Compare periodic (CAG)10/(CAG)20/(CAG)31/(CAG)47 RNAs and length-matched
scrambled controls. This notebook deliberately keeps **length** and
**sequence-scramble controls** within RQ2; no separate mislabeled RQ3
projection notebook.

Plots use the wide CSV exports. Statistical comparisons must use independent
scramble sequences rather than counting repeated trajectories as independent.

In [ ]:
from pathlib import Path
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Works when Jupyter starts from the repository or analysis/notebooks/.
here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"


In [ ]:
from src.plots.data import independent_units
from src.plots.figures import (
    plot_cluster_bins, plot_ranked_clusters, plot_metric_points
)
PERIODIC_SWEEP = "51a_repeat_number"
SCRAMBLED_SWEEP = "51b_repeat_number_rand"
DENSITY = 0.01
STIFFNESS = 2.5
periodic = filter_rows(load_sweep(OUTPUT_ROOT, PERIODIC_SWEEP, "summary"),
                       density=DENSITY, angle_k_ppp=STIFFNESS)
scrambled = filter_rows(load_sweep(OUTPUT_ROOT, SCRAMBLED_SWEEP, "summary"),
                        density=DENSITY, angle_k_ppp=STIFFNESS)
periodic_reps = filter_rows(load_sweep(OUTPUT_ROOT, PERIODIC_SWEEP, "replicates"),
                            density=DENSITY, angle_k_ppp=STIFFNESS)
scrambled_reps = filter_rows(load_sweep(OUTPUT_ROOT, SCRAMBLED_SWEEP, "replicates"),
                             density=DENSITY, angle_k_ppp=STIFFNESS)
periodic["sequence_class"] = "repeat"
scrambled["sequence_class"] = "scrambled"
periodic_reps["sequence_class"] = "repeat"
scrambled_reps["sequence_class"] = "scrambled"
summaries = pd.concat([periodic, scrambled], ignore_index=True)
independent = independent_units(pd.concat(
    [periodic_reps, scrambled_reps], ignore_index=True
))
print(f"Periodic trajectories: {len(periodic_reps)}; "
      f"independent scrambled units: {(independent.sequence_class == 'scrambled').sum()}")

## Length dependence: particle-percentage cluster classes

Bars are grouped by RNA repeat number for each attraction strength, with
monomers shown separately. No extra RDF or 1D count-profile panels.

In [ ]:
fig, axes = panels_by(
    periodic, by="E_att",
    plot_fn=partial(plot_cluster_bins, scheme="particle_percentage",
                    x="repeat_count", style=STYLE),
    style=STYLE, title="Cluster-size composition across repeat lengths",
)
plt.show()

## Periodic versus scrambled: connectivity distributions

The notebook uses one observation per independent scrambled sequence.
`seaborn` provides boxplots and replicate dots; colors/labels remain
controlled by our shared PlotStyle.

In [ ]:
palette = {"repeat": "#286D9E", "scrambled": "#CB7742"}
fig, axes = panels_by(
    independent, by="repeat_count",
    plot_fn=partial(plot_metric_points, metric="mean_edge_weight",
                    hue="sequence_class", palette=palette, style=STYLE),
    style=STYLE, title="Mean qualifying molecule-pair edge weight",
)
plt.show()

## Ranked cluster composition

Visualize the largest, second, and third multimer components, other multimers,
and all monomers. The categories sum to 100% of particles.

In [ ]:
fig, axes = panels_by(
    periodic, by="repeat_count",
    plot_fn=partial(plot_ranked_clusters, style=STYLE),
    style=STYLE, title="Ranked cluster material: periodic RNA",
)
plt.show()

## Representative final-frame XY projections

Optional: the same clean projection API as reentrance, but **rows are RNA
repeat lengths**, not backbone stiffness values.

In [ ]:
SHOW_PROJECTIONS = False
if SHOW_PROJECTIONS:
    from src.plots.projections import plot_projection_sweep
    GSD_ROOT = ROOT.parent / "validation" / "bulk" / "output_dir"
    fig, axes = plot_projection_sweep(
        GSD_ROOT / PERIODIC_SWEEP, row_by="repeat_count",
        density=DENSITY, attractions=(2, 4, 8), style=STYLE,
        title="RQ2 · RNA length dependence",
    )
    plt.show()

## Statistical follow-up

Holm-adjusted permutation tests belong here, not in the production simulation
analysis pipeline. Tests should compare replicate-level edge-weight means,
using each scrambled RNA sequence as an independent unit. Add a clearly
defined family of comparisons before reporting adjusted p-values.